In [1]:
import pandas as pd
from pathlib import Path

In [2]:
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    base_dir = current_dir.parent
else:
    base_dir = current_dir

DATA_DIR = base_dir / "data"

train = pd.read_csv(DATA_DIR / "train-test.csv", parse_dates=["date"])
val = pd.read_csv(DATA_DIR / "validation.csv", parse_dates=["date"])
dec = pd.read_csv(DATA_DIR / "december-chart-inputs.csv", parse_dates=["date"])
template = pd.read_csv(DATA_DIR / "validation-predictions-template.csv")

In [3]:
for name, df in [("train", train), ("val", val), ("dec", dec)]:
    print(f"{name} shape: {df.shape}")

train shape: (48000, 14)
val shape: (12000, 13)
dec shape: (31, 7)


In [4]:
# Data Types
train.dtypes

load_id                 object
pickup                  object
delivery                object
pickup_lat             float64
pickup_lon             float64
delivery_lat           float64
delivery_lon           float64
distance               float64
equipment               object
weight                 float64
date            datetime64[ns]
market_index           float64
quote_signal           float64
posted_rate            float64
dtype: object

In [5]:
# Null Counts
train.isna().sum()

load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64

In [6]:
# Date Range train vs validation
print(train["date"].min(), "->", train["date"].max()) 
print(val["date"].min(), "->", val["date"].max())

2025-01-01 00:00:00 -> 2025-10-31 00:00:00
2025-11-01 00:00:00 -> 2025-12-31 00:00:00


In [7]:
# posted_rate distribution
train['posted_rate'].describe()

count    48000.000000
mean      2373.980682
std       1486.493245
min         57.220000
25%       1251.555000
50%       2030.760000
75%       3330.750000
max      25533.000000
Name: posted_rate, dtype: float64

In [8]:
# Outliers
print("Potential Outlier :",(train['posted_rate'] <= 0).sum())

Potential Outlier : 0


In [9]:
# Top 10 posted_rate
print("Top 10 highest posted_rate:")
train.nlargest(10, "posted_rate")[["load_id", "distance", "equipment", "posted_rate"]]

Top 10 highest posted_rate:


,load_id,distance,equipment,posted_rate
12184,TR-012185,2829.8,Dry Van,25533.00
37764,TR-037765,2810.9,Dry Van,24294.98
1466,TR-001467,2786.0,Reefer,24140.21
17371,TR-017372,2777.6,Reefer,23662.71
26235,TR-026236,2552.5,Dry Van,23580.42
28219,TR-028220,2423.3,Dry Van,22755.66
3353,TR-003354,2527.9,Reefer,22534.65
40171,TR-040172,2283.4,Dry Van,20361.89
40096,TR-040097,1760.3,Reefer,20132.37
47298,TR-047299,2979.1,Reefer,19110.30


In [10]:
# Bottom 10 posted_rate
print("Bottom 10 lowest posted_rate:")
train.nsmallest(10, "posted_rate")[["load_id", "distance", "equipment", "posted_rate"]]

Bottom 10 lowest posted_rate:


,load_id,distance,equipment,posted_rate
5961,TR-005962,105.5,Dry Van,57.22
30705,TR-030706,114.4,Dry Van,59.68
14990,TR-014991,93.9,Reefer,63.36
872,TR-000873,144.0,Dry Van,67.80
4674,TR-004675,149.3,Reefer,75.19
9137,TR-009138,105.0,Reefer,76.54
34343,TR-034344,70.0,Dry Van,82.38
10959,TR-010960,122.6,Reefer,94.72
18954,TR-018955,228.7,Dry Van,101.63
11236,TR-011237,228.3,Dry Van,103.15


In [11]:
# Leakage Check: Correlation of posted_rate with market_index and quote_signal
train[["market_index", "quote_signal", "posted_rate", "distance", "weight"]].corr()["posted_rate"].sort_values(ascending=False)

posted_rate     1.000000
distance        0.908519
weight          0.034840
market_index    0.034165
quote_signal   -0.039858
Name: posted_rate, dtype: float64

In [12]:
# Equipment Value Counts
train["equipment"].value_counts()

equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64

In [13]:
# Equipment value NOT in train but in validation
set(val["equipment"].unique()) - set(train["equipment"].unique())

set()

In [14]:
# Equipment value in december-chart-inputs
dec["equipment"].value_counts()

equipment
Dry Van    31
Name: count, dtype: int64

In [15]:
# Unique pickup and delivery locations in train
print("Unique pickup locations in train :", train['pickup'].nunique())
print("Unique delivery locations in train :", train['delivery'].nunique())

Unique pickup locations in train : 64
Unique delivery locations in train : 64


In [16]:
# Check if december's fixed lane (Lexington -> Fort Wayne) appears in train-test
fixed_lane_mask = (train["pickup"] == "Lexington") & (train["delivery"] == "Fort Wayne")
print(f"Lexington -> Fort Wayne appears {fixed_lane_mask.sum()} times in train_test")

Lexington -> Fort Wayne appears 32 times in train_test


In [17]:
# Template / ID Format Check
template.head(5)

,load_id,predicted_rate
0,TE-000001,NaN
1,TE-000002,NaN
2,TE-000003,NaN
3,TE-000004,NaN
4,TE-000005,NaN


In [18]:
dec.columns.tolist()

['pickup',
 'delivery',
 'distance',
 'equipment',
 'weight',
 'date',
 'predicted_rate']

In [19]:
print("Do template and validation.csv load_ids match exactly : ", set(template["load_id"]) == set(val["load_id"]))

Do template and validation.csv load_ids match exactly :  True
